In [ ]:

import pandas as pd

# Load the train and test datasets
train_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\train.csv'
test_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the train and test datasets
print("Train dataset head:")
print(train_df.head())
print("\nTest dataset head:")
print(test_df.head())

# Display basic information about the train and test datasets
print("\nTrain dataset info:")
train_df.info()
print("\nTest dataset info:")
test_df.info()


Train dataset head:
                     Disease Fever  ... Cholesterol Level Outcome Variable
0                Ebola Virus   Yes  ...            Normal         Positive
1  Conjunctivitis (Pink Eye)    No  ...              High         Positive
2               Pancreatitis    No  ...              High         Positive
3               Pancreatitis   Yes  ...            Normal         Negative
4            Hyperthyroidism   Yes  ...            Normal         Negative

[5 rows x 10 columns]

Test dataset head:
                                        Disease  ... Outcome Variable
0                                Hypothyroidism  ...         Positive
1                                   Tonsillitis  ...         Positive
2  Chronic Obstructive Pulmonary Disease (COPD)  ...         Positive
3                                  Lyme Disease  ...         Positive
4                                      Diabetes  ...         Positive

[5 rows x 10 columns]

Train dataset info:
<class 'pandas.core.fra

In [ ]:


# One-hot encoding categorical features (excluding 'Disease')
categorical_features = ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Gender', 'Blood Pressure', 'Cholesterol Level']

# Perform one-hot encoding
train_encoded = pd.get_dummies(train_df, columns=categorical_features, drop_first=True)
test_encoded = pd.get_dummies(test_df, columns=categorical_features, drop_first=True)

# Display the first few rows of the encoded train and test datasets
print("Encoded Train dataset head:")
print(train_encoded.head())
print("\nEncoded Test dataset head:")
print(test_encoded.head())

# Ensure the columns are the same in train and test datasets
train_columns = set(train_encoded.columns)
test_columns = set(test_encoded.columns)

# Add missing columns in test dataset
missing_columns = train_columns - test_columns
for col in missing_columns:
    test_encoded[col] = 0

# Reorder columns in test dataset to match train dataset
test_encoded = test_encoded[train_encoded.columns]

# Display the final column order
print("\nFinal column order for train and test datasets:")
print(train_encoded.columns)
print(test_encoded.columns)



Encoded Train dataset head:
                     Disease  ...  Cholesterol Level_Normal
0                Ebola Virus  ...                      True
1  Conjunctivitis (Pink Eye)  ...                     False
2               Pancreatitis  ...                     False
3               Pancreatitis  ...                      True
4            Hyperthyroidism  ...                      True

[5 rows x 12 columns]

Encoded Test dataset head:
                                        Disease  ...  Cholesterol Level_Normal
0                                Hypothyroidism  ...                     False
1                                   Tonsillitis  ...                     False
2  Chronic Obstructive Pulmonary Disease (COPD)  ...                     False
3                                  Lyme Disease  ...                     False
4                                      Diabetes  ...                      True

[5 rows x 12 columns]

Final column order for train and test datasets:
Index(['Disease

In [ ]:

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Define the logistic regression model for feature selection
model = LogisticRegression(max_iter=1000)

# Define the RFE model to select the top 5 features
rfe = RFE(model, n_features_to_select=5, step=1)

# Fit the RFE model on the training data
X_train = train_encoded.drop(columns=['Disease', 'Outcome Variable'])
y_train = train_encoded['Outcome Variable']
rfe.fit(X_train, y_train)

# Get the selected features
selected_features = X_train.columns[rfe.support_]
print("Selected features:", selected_features)

# Create a new dataset with only the selected features
X_train_selected = X_train[selected_features]
X_test_selected = test_encoded[selected_features]

# Display the first few rows of the selected features in the train and test datasets
print("\nTrain dataset with selected features head:")
print(X_train_selected.head())
print("\nTest dataset with selected features head:")
print(X_test_selected.head())


Selected features: Index(['Fever_Yes', 'Fatigue_Yes', 'Gender_Male', 'Cholesterol Level_Low',
       'Cholesterol Level_Normal'],
      dtype='object')

Train dataset with selected features head:
   Fever_Yes  Fatigue_Yes  ...  Cholesterol Level_Low  Cholesterol Level_Normal
0       True         True  ...                  False                      True
1      False        False  ...                  False                     False
2      False         True  ...                  False                     False
3       True        False  ...                  False                      True
4       True         True  ...                  False                      True

[5 rows x 5 columns]

Test dataset with selected features head:
   Fever_Yes  Fatigue_Yes  ...  Cholesterol Level_Low  Cholesterol Level_Normal
0       True         True  ...                  False                     False
1       True         True  ...                  False                     False
2       True       

In [ ]:


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the logistic regression model
logreg_all = LogisticRegression(max_iter=1000)

# Fit the model on the full feature set
X_train_all = train_encoded.drop(columns=['Disease', 'Outcome Variable'])
y_train = train_encoded['Outcome Variable']
logreg_all.fit(X_train_all, y_train)

# Predict on the test set
X_test_all = test_encoded.drop(columns=['Disease', 'Outcome Variable'])
y_test = test_encoded['Outcome Variable']
y_pred_all = logreg_all.predict(X_test_all)

# Evaluate the model
accuracy_all = accuracy_score(y_test, y_pred_all)
precision_all = precision_score(y_test, y_pred_all, pos_label='Positive')
recall_all = recall_score(y_test, y_pred_all, pos_label='Positive')
f1_all = f1_score(y_test, y_pred_all, pos_label='Positive')

print("Model 1 (All Features) Performance:")
print(f"Accuracy: {accuracy_all:.4f}")
print(f"Precision: {precision_all:.4f}")
print(f"Recall: {recall_all:.4f}")
print(f"F1 Score: {f1_all:.4f}")

# Define the logistic regression model for the selected features
logreg_selected = LogisticRegression(max_iter=1000)

# Fit the model on the selected feature set
logreg_selected.fit(X_train_selected, y_train)

# Predict on the test set
y_pred_selected = logreg_selected.predict(X_test_selected)

# Evaluate the model
accuracy_selected = accuracy_score(y_test, y_pred_selected)
precision_selected = precision_score(y_test, y_pred_selected, pos_label='Positive')
recall_selected = recall_score(y_test, y_pred_selected, pos_label='Positive')
f1_selected = f1_score(y_test, y_pred_selected, pos_label='Positive')

print("\nModel 2 (Selected Features) Performance:")
print(f"Accuracy: {accuracy_selected:.4f}")
print(f"Precision: {precision_selected:.4f}")
print(f"Recall: {recall_selected:.4f}")
print(f"F1 Score: {f1_selected:.4f}")



Model 1 (All Features) Performance:
Accuracy: 0.5714
Precision: 0.6316
Recall: 0.6000
F1 Score: 0.6154

Model 2 (Selected Features) Performance:
Accuracy: 0.5714
Precision: 0.6316
Recall: 0.6000
F1 Score: 0.6154
